# "THE POUR IS RIGHT" - Wine Pricer Capstone

## Week 8 Order of play

DAY 1: Modal and the SpecialistAgent  
DAY 2: RAG with Qdrant, FrontierAgent, Ensemble  
DAY 3: ScannerAgent, MessagingAgent  
DAY 4: The planning agent  
DAY 5: The user interface  

## DAY 3: find deals, and tell the user about them

Today we build the two agents at either end of the system: one that finds wines for sale, and one that sends you a message about a good one. This follows `week8/day3.ipynb`.

## Where the deals come from

Ed's scanner read deals from DealNews RSS feeds, which carry no wine. Ours reads three sources:

| Source | How | Market |
|---|---|---|
| WTSO (Wines Till Sold Out) | its home page | US, dollars |
| Last Bottle | its home page | US, dollars |
| HotUKDeals, wine tag | RSS feed | UK, pounds |

The two US shops show one or two offers at a time, in the same market our models learned. The UK feed adds volume, and its prices are converted to dollars.

The sources are listed at the top of `agents/deals.py`. Adding a shop later, for example in Kenya or South Africa, means adding a line there.

## Costs

| Step | Cost |
|---|---|
| Reading the sources | free |
| One scan with GPT-5-mini | about 1 cent |
| One message written by Claude Sonnet 5 | under 1 cent |
| Sending a Telegram message | free |

In [ ]:
import os
import logging
from dotenv import load_dotenv
from openai import OpenAI
import requests
from agents.deals import ScrapedDeal, DealSelection, gbp_to_usd

load_dotenv(override=True)
openai = OpenAI()

# Read the sources

`ScrapedDeal.fetch()` reads each source. For the RSS feed it takes the title and summary of the 10 newest posts. For the two shops it reads the home page and keeps the stretch of text around the first price, which is where the current offer is.

If one source fails, for example because a site is down or has been redesigned, it is skipped with a warning and the others carry on.

In [ ]:
deals = ScrapedDeal.fetch(show_progress=True)

In [ ]:
print(f"{len(deals)} deals")
deals

In [ ]:
# One of the US shop pages

print(deals[0].describe())

In [ ]:
# And one from the UK feed

print(deals[-1].describe())

The shop pages are messy: menus, buttons and other text around the offer. Rather than writing a parser for each site, which would break the next time a site is redesigned, we hand this text to an LLM and ask it to pull out the deals. That is the scanner's job.

### A note on the page addresses

A shop's home page has the same address every day, while its offer changes. So each page deal's URL gets a fingerprint of its text after a `#`. A new offer gets a new fingerprint, which lets the agents tell today's deal from yesterday's.

# Ask GPT-5-mini to pick and summarise the deals

The scanner sends all the deals to GPT-5-mini, using **structured outputs**: we give it a Pydantic class, `DealSelection` in `agents/deals.py`, and the reply must fit it. Each deal comes back as a description of the wine, a price per bottle, a currency and a URL.

Open `agents/deals.py` and look at the `Field(description=...)` text on each field of `Deal`. The model reads those descriptions as part of its instructions.

The system prompt adds rules learned from testing:

- only wine, not spirits or glasses
- the price of **one** bottle: a case of 6 is divided by 6
- always include the US shops' offers. Without this, the ten UK deals crowded them out.
- use only facts from the text. Without this, the model invented tasting notes, as the summary model did on day 2 of week 6.

In [ ]:
from agents.scanner_agent import ScannerAgent

print(ScannerAgent.SYSTEM_PROMPT)

In [ ]:
scanner = ScannerAgent()
user_prompt = scanner.make_user_prompt(deals)
print(user_prompt[:2000])

In [ ]:
messages = [{"role": "system", "content": ScannerAgent.SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

response = openai.chat.completions.parse(model="gpt-5-mini", messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed

for deal in results.deals:
    print(f"{deal.price} {deal.currency}  {deal.url}")
    print(deal.product_description)
    print()

## Pounds to dollars

Our models learned US prices, so the UK prices are converted to dollars. `gbp_to_usd()` gets today's rate from a free exchange-rate service, and falls back to a fixed rate if the service can't be reached.

Bear in mind that conversion is only half the story. Wine costs different amounts in different countries for reasons beyond the exchange rate, such as taxes, so a UK price in dollars is a rougher comparison than a US price.

In [ ]:
rate = gbp_to_usd()
print(f"1 GBP = {rate:.3f} USD")

# The ScannerAgent

The same steps in one call: read the sources, leave out any deal already in memory, ask GPT-5-mini to choose, and convert prices to dollars. `memory` is the list of deals we have already told the user about; on day 4 it will stop the same deal being sent twice.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
for deal in result.deals:
    print(f"${deal.price:.2f}  {deal.url}")
    print(deal.product_description)
    print()

`test_scan()` returns three fixed deals without reading any website or calling any model. It is useful when building the next stages, so you are not paying for a scan every time.

In [ ]:
scanner.test_scan()

# Telegram

The bot `@waruts_wine_bot` sends messages to your chat. The two values it needs are in `.env`:

- `TELEGRAM_BOT_TOKEN`: the bot's password, from BotFather
- `TELEGRAM_CHAT_ID`: the number of your chat with the bot

Sending a message is one web request to Telegram. The cells below send real messages to your phone.

In [ ]:
telegram_token = os.getenv("TELEGRAM_BOT_TOKEN")
telegram_chat_id = os.getenv("TELEGRAM_CHAT_ID")

print(f"Telegram token found: {bool(telegram_token)}")
print(f"Telegram chat ID found: {bool(telegram_chat_id)}")

In [ ]:
def push(message):
    print(f"Push: {message}")
    url = f"https://api.telegram.org/bot{telegram_token}/sendMessage"
    response = requests.post(url, data={"chat_id": telegram_chat_id, "text": message}, timeout=30)
    print(f"Telegram says ok: {response.json().get('ok')}")

In [ ]:
push("MASSIVE WINE DEAL!!")

# The MessagingAgent

The same thing, as an agent. It has two ways to send an alert:

- `alert()` builds the message from a template, for free.
- `notify()` asks Claude Sonnet 5 to write a short, lively message about the deal first.

In [ ]:
from agents.messaging_agent import MessagingAgent

messenger = MessagingAgent()
messenger.push("SUCH A MASSIVE WINE DEAL!!")

In [ ]:
messenger.notify(
    "Decades 5 Cabernet Sauvignon 2023 from Napa Valley, California, made by winemaker Jean Hoefliger. 96 points.",
    49.0,
    110.0,
    "https://www.lastbottlewines.com/",
)